# **02 — Pré-processamento**

Neste notebook, foi realizada a limpeza de dados faltantes, binarização da variável alvo (`TARGET`), criação de novas variáveis de negócio (*Feature Engineering*), tratamento de incoerências e exportação do dataset final para a etapa de modelagem.

In [1]:
from google.colab import drive
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np

# semente fixa
RANDOM_STATE = 42

# montar o Google Drive
drive.mount('/content/drive')

# configuração caminhos base
drive_root = Path('/content/drive/MyDrive')
app_paths = list(drive_root.rglob('application_record.csv'))
cred_paths = list(drive_root.rglob('credit_record.csv'))

if app_paths and cred_paths:
    RAW_APPLICATION = app_paths[0]
    RAW_CREDIT = cred_paths[0]
    PROCESSED = RAW_APPLICATION.parent / "processed"
    PROCESSED.mkdir(parents=True, exist_ok=True)
    print("✓ Arquivos encontrados com sucesso no seu Google Drive!")
else:
    # Fallback caso não encontre (usando o caminho conhecido)
    RAW_APPLICATION = Path("/content/drive/MyDrive/Pós Tech/Tech Challange/BaseDadosFase2/application_record.csv")
    RAW_CREDIT = Path("/content/drive/MyDrive/Pós Tech/Tech Challange/BaseDadosFase2/credit_record.csv")
    PROCESSED = Path("/content/drive/MyDrive/Pós Tech/Tech Challange/BaseDadosFase2/processed")

TARGET = "TARGET"
pd.set_option("display.max_columns", None)

# carregamento dados brutos
application = pd.read_csv(RAW_APPLICATION)
credit = pd.read_csv(RAW_CREDIT)
print("Dados carregados com sucesso!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Arquivos encontrados com sucesso no seu Google Drive!
Dados carregados com sucesso!


# **1. Dados faltantes**

In [2]:
# Mapeamento e contagem de valores nulos
nulos = application.isna().sum()
pct_nulos = (nulos / len(application) * 100).round(2)
df_nulos = pd.DataFrame({"Total Nulos": nulos, "Percentual (%)": pct_nulos}).query("`Total Nulos` > 0")

print("=== DIAGNÓSTICO DE VALORES FALTANTES ===")
display(df_nulos)

# Imputação da categoria "Desconhecido" para OCCUPATION_TYPE
application['OCCUPATION_TYPE'] = application['OCCUPATION_TYPE'].fillna("Desconhecido")
print("\n✓ Tratamento concluído: Valores nulos imputados como 'Desconhecido'.")

=== DIAGNÓSTICO DE VALORES FALTANTES ===


,Total Nulos,Percentual (%)
OCCUPATION_TYPE,134203,30.6



✓ Tratamento concluído: Valores nulos imputados como 'Desconhecido'.


### Análise— Tratamento de Nulos
Apenas a variável `OCCUPATION_TYPE` continha valores ausentes (~30% da base). Optado por imputar a categoria **`Desconhecido`** em vez de remover as linhas. Isso preserva todo o volume de dados cadastrais e permite ao modelo identificar se a omissão da ocupação possui correlação com o risco de crédito.

# **2. Definição da variável alvo**


In [3]:
# Mapear atrasos >= 60 dias (status 2, 3, 4, 5) no histórico de crédito
credit['IS_BAD'] = credit['STATUS'].isin(['2', '3', '4', '5']).astype(int)

# Agrupar por ID para capturar o maior risco histórico
target_df = credit.groupby('ID')['IS_BAD'].max().reset_index()
target_df.rename(columns={'IS_BAD': 'TARGET'}, inplace=True)

# Evitar duplicação da coluna TARGET caso a célula seja reexecutada
if 'TARGET' in application.columns:
    application = application.drop(columns=['TARGET'])

# Cruzamento dos cadastros com o alvo mapeado
application = application.merge(target_df, on='ID', how='inner')

print("✓ Variável TARGET criada com sucesso!")
print(application['TARGET'].value_counts(normalize=True) * 100)

✓ Variável TARGET criada com sucesso!
TARGET
0    98.310338
1     1.689662
Name: proportion, dtype: float64


### Análise — Variável Alvo
Definido como **Mau Pagador (`TARGET = 1`)** qualquer cliente com atrasos de pagamento $\ge 60$ dias (`STATUS` = 3, 4 ou 5).  Atrasos menores (como 30 dias) são frequentemente pontuais ou operacionais. A partir de 60 dias caracteriza-se o risco real de inadimplência e necessidade de provisionamento bancário.


# **3. Normalização / padronização**

In [4]:
from sklearn.preprocessing import StandardScaler

# Declaramos o scaler que será utilizado
scaler = StandardScaler()
print("✓ StandardScaler importado com sucesso!")

✓ StandardScaler importado com sucesso!


### Decisão — Aplicação do Scaler
**Importante:** A transformação de escala com `StandardScaler` será aplicada **exclusivamente dentro de um Pipeline no Notebook 03 (Modelagem)** durante a validação cruzada.

**Justificativa Técnica:** Ajustar o scaler na base inteira antes do split de treino e teste causaria **Vazamento de Dados (*Data Leakage*)**, invalidando os resultados de teste.

# **4. Feature engineering**

In [5]:
# 1. Correção e conversão de Idade (Anos positivos)
application['AGE'] = (-application['DAYS_BIRTH'] / 365.25).round(1)

# 2. Correção de Tempo de Emprego e tratamento do código de Aposentados (365243)
application['DAYS_EMPLOYED_CLEAN'] = application['DAYS_EMPLOYED'].apply(lambda x: 0 if x > 0 else np.abs(x))
application['YEARS_EMPLOYED'] = (application['DAYS_EMPLOYED_CLEAN'] / 365.25).round(1)

# 3. Criação da Renda Per Capita Familiar
application['INCOME_PER_PERSON'] = (application['AMT_INCOME_TOTAL'] / application['CNT_FAM_MEMBERS']).round(2)

# 4. Criação de Faixas Etárias (Binned Feature)
application['AGE_GROUP'] = pd.cut(
    application['AGE'],
    bins=[20, 30, 40, 50, 60, 70],
    labels=['20-30', '30-40', '40-50', '50-60', '60-70']
)

print("=== NOVAS VARIÁVEIS GERADAS ===")
display(application[['AMT_INCOME_TOTAL', 'CNT_FAM_MEMBERS', 'INCOME_PER_PERSON', 'AGE', 'AGE_GROUP', 'YEARS_EMPLOYED']].head(3))


=== NOVAS VARIÁVEIS GERADAS ===


,AMT_INCOME_TOTAL,CNT_FAM_MEMBERS,INCOME_PER_PERSON,AGE,AGE_GROUP,YEARS_EMPLOYED
0,427500.0,2.0,213750.0,32.9,30-40,12.4
1,427500.0,2.0,213750.0,32.9,30-40,12.4
2,112500.0,2.0,56250.0,58.8,50-60,3.1


### Decisão — Feature Engineering
* **`INCOME_PER_PERSON` (Renda Per Capita):** A renda bruta isolada não reflete o comprometimento real do orçamento quando o cliente possui dependentes. A renda por pessoa oferece maior precisão no cálculo do risco financeiro.
* **`AGE_GROUP` (Faixa Etária):** Permite ao modelo capturar padrões não-lineares associados à maturidade socioeconômica do cliente.
* **Tratamento de `DAYS_EMPLOYED`:** O código `365243` atribuído a aposentados/desempregados foi zerado para não distorcer a escala de tempo de serviço.


# **5. Salvar dataset tratado**

In [6]:
# Selecionar variáveis categóricas para One-Hot Encoding
categorical_cols = [
    'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS',
    'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'AGE_GROUP', 'CODE_GENDER',
    'FLAG_OWN_CAR', 'FLAG_OWN_REALTY'
]

# Aplicar One-Hot Encoding (drop_first=True para evitar multicolinearidade)
df_processed = pd.get_dummies(application, columns=categorical_cols, drop_first=True)

# Converter booleanos em inteiros (0 ou 1)
bool_cols = df_processed.select_dtypes(include=['bool']).columns
df_processed[bool_cols] = df_processed[bool_cols].astype(int)

# Salvar no diretório processed
OUTPUT_FILE = PROCESSED/ "dataset_tratado.csv"
df_processed.to_csv(OUTPUT_FILE, index=False)

print(f"✓ Dataset tratado exportado com sucesso para: {OUTPUT_FILE}")
print(f"✓ Formato final do dataset tratado: {df_processed.shape[0]} linhas × {df_processed.shape[1]} colunas")


✓ Dataset tratado exportado com sucesso para: /content/drive/MyDrive/Pós Tech/Tech Challange/BaseDadosFase2/processed/dataset_tratado.csv
✓ Formato final do dataset tratado: 36457 linhas × 57 colunas


In [7]:
import os
from datetime import datetime

# Caminho do arquivo de saída definido na célula anterior
caminho_arquivo = PROCESSED / "dataset_tratado.csv"

if caminho_arquivo.exists():
    # Obter estatísticas do arquivo
    estatisticas = os.stat(caminho_arquivo)

    # Converter o timestamp de modificação para um formato legível
    data_modificacao = datetime.fromtimestamp(estatisticas.st_mtime)
    tamanho_mb = estatisticas.st_size / (1024 * 1024)

    print("=== STATUS DO ARQUIVO EXPORTADO ===")
    print(f"Caminho: {caminho_arquivo}")
    print(f"Última modificação: {data_modificacao.strftime('%d/%m/%Y %H:%M:%S')}")
    print(f"Tamanho do arquivo: {tamanho_mb:.2f} MB")
else:
    print(f"❌ O arquivo ainda não foi criado no caminho: {caminho_arquivo}")

=== STATUS DO ARQUIVO EXPORTADO ===
Caminho: /content/drive/MyDrive/Pós Tech/Tech Challange/BaseDadosFase2/processed/dataset_tratado.csv
Última modificação: 10/10/2026 22:20:59
Tamanho do arquivo: 5.27 MB


In [8]:
from google.colab import files

# Caminho do arquivo tratado no Drive
caminho_arquivo = PROCESSED / "dataset_tratado.csv"

# Executa o download para a sua máquina local
if caminho_arquivo.exists():
    files.download(str(caminho_arquivo))
else:
    print("❌ Arquivo não encontrado para download. Por favor, execute as células anteriores primeiro.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Decisão — Salvamento do Dataset
O dataset `dataset_tratado.csv` foi exportado para a pasta `data/processed/`. Isso garante a reprodutibilidade, evita refazer o pré-processamento a cada execução e permite que o **Notebook 03 (Modelagem)** recarregue diretamente a base tratada de forma independente.